<a href="https://colab.research.google.com/github/mark92233/FUNDAI-Laboratories-Ando/blob/main/Lab4_logic_KR_Ando.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 4:  Logic and Knowledge Representation Using Python

## Fundamentals of artificial Inteliggence

**Name:** Ando, Mark John S
**Course:** BSCSAI
**Section:** 2A  
**Date:** September 15, 2026
**GitHub URL:** https://github.com/mark92233/FUNDAI-Laboratories-09282.git


## Description
This laboratory uses Python and Sympy to perform truth table generation
satisfiability checking, theorem proving, and logical deduction.



In [3]:
from sympy import symbols, And, Or, Not, Implies, Equivalent, satisfiable
from itertools import product

In [4]:
P, Q, R = symbols('P Q R')

In [6]:
def print_truth_table(expression, symbol_list):
  header = [str(s) for s in symbol_list] + [str(expression)]
  print(" | ".join(header))
  print("-" * (5 * len(header)))

  for values in product([False, True], repeat=len(symbol_list)):
    mapping = dict(zip(symbol_list, values))
    result = bool(expression.subs(mapping))
    row = [str(v) for v in values] + [str(result)]
    print(" | ".join(row))

  print()

In [7]:
print("Negation: NOT P")
print_truth_table(Not(P), [P])

print("Conjunction: P AND Q")
print_truth_table(And(P, Q), [P, Q])

print("Disjunction: P OR Q")
print_truth_table(Or(P, Q), [P, Q])

print("Implication: P -> Q")
print_truth_table(Implies(P, Q), [P, Q])

print("Biconditional: P <-> Q")
print_truth_table(Equivalent(P, Q), [P, Q])

Negation: NOT P
P | ~P
----------
False | True
True | False

Conjunction: P AND Q
P | Q | P & Q
---------------
False | False | False
False | True | False
True | False | False
True | True | True

Disjunction: P OR Q
P | Q | P | Q
---------------
False | False | False
False | True | True
True | False | True
True | True | True

Implication: P -> Q
P | Q | Implies(P, Q)
---------------
False | False | True
False | True | True
True | False | False
True | True | True

Biconditional: P <-> Q
P | Q | Equivalent(P, Q)
---------------
False | False | True
False | True | False
True | False | False
True | True | True



In [8]:
def is_tautology(expression, symbol_list):
  for values in product([False, True], repeat=len(symbol_list)):
    mapping = dict(zip(symbol_list, values))
    if not bool(expression.subs(mapping)):
      return False
  return True

In [9]:
law_of_excluded_middle = Or(P, Not(P))
contradiction = And(P, Not(P))
simple_implication = Implies(P, Q)

print("P OR NOT P is tautology: ", is_tautology(law_of_excluded_middle, [P]))
print("P AND NOT P is tautology: ", is_tautology(contradiction, [P]))
print("P -> Q is tautology: ", is_tautology(simple_implication, [P, Q]))

P OR NOT P is tautology:  True
P AND NOT P is tautology:  False
P -> Q is tautology:  False


In [10]:
def is_satisfiable(expression):
  return satisfiable(expression) is not False

print("P AND NOT P is satisfiable: ", is_satisfiable(law_of_excluded_middle))
print("P OR NOT P is satisfiable: ", is_satisfiable(contradiction))
print("P -> Q is satisfiable: ", is_satisfiable(simple_implication))

P AND NOT P is satisfiable:  True
P OR NOT P is satisfiable:  False
P -> Q is satisfiable:  True


In [17]:
def to_conjunction(kb):
  if isinstance(kb, list):
    return And(*kb)
  return kb

def kb_entails(kb, conclusion):
  kb_expression = to_conjunction(kb)
  counter_check = And(kb_expression, Not(conclusion))
  return is_satisfiable(counter_check) is False

def check_entailment(kb, conclusion, label="Query"):
  holds = kb_entails(kb, conclusion)
  kb_expression = to_conjunction(kb)
  counterexample = satisfiable(And(kb_expression, Not(conclusion)))

  print(label)

  if holds:
    print("Results: Entailment holds.")
  else:
    print("Result: Entailment does not hold." )
    print("Counterexample model: ", counterexample)

  print("-" * 100)
  return holds


In [18]:
Rain, Wet = symbols("Rain Wet")

kb_rain = [
    Implies(Rain, Wet),
    Rain
]

check_entailment(kb_rain, Wet, "Theorem Proving: Rain example")

Theorem Proving: Rain example
Results: Entailment holds.
----------------------------------------------------------------------------------------------------


True

In [19]:
kb_invalid = [
    Implies(Rain, Wet),
    Wet
]

check_entailment(kb_invalid, Rain, "Invalid Inference: Affirming the consequent.")

Invalid Inference: Affirming the consequent.
Result: Entailment does not hold.
Counterexample model:  {Wet: True, Rain: False}
----------------------------------------------------------------------------------------------------


False

In [20]:
print("Logical Deduction Rules")
print("=" * 100)

#Modus ponens
check_entailment(
    [P, Implies(P, Q)],
    Q,
    "Modus Ponens: P P->Q, Therefore Q"
)

#Modus Tollens
check_entailment(
    [Not(Q), Implies(P, Q)],
    Not(P),
    "Modus Tollens: NOT Q P->Q Therefore NOT P"
)

Logical Deduction Rules
Modus Ponens: P P->Q, Therefore Q
Results: Entailment holds.
----------------------------------------------------------------------------------------------------
Modus Tollens: NOT Q P->Q Therefore NOT P
Results: Entailment holds.
----------------------------------------------------------------------------------------------------


True

## Grounded First-Order Logic Example

Full First-Order Logic includes objects and quantifiers.
For this laboratory, we demonstrate a simple grounded FOL example by converting FOL atoms into propositional symbols.

English:

* All humans are mortal.
* Socrates is human.
* Therefore, Socrates is mortal.

Grounded propositional form:

* Human_Socrates -> Mortal_Socrates


In [21]:
Human_Socrates, Mortal_Socrates = symbols("Human_Socrates Mortal_Socrates")

kb_socrates = [
    Implies(Human_Socrates, Mortal_Socrates),
    Human_Socrates
]

check_entailment(
    kb_socrates,
    Mortal_Socrates,
    "Grounded FOL: Socrates is mortal"
)

Grounded FOL: Socrates is mortal
Results: Entailment holds.
----------------------------------------------------------------------------------------------------


True

In [27]:
def make_human_mortal_kb(constants):
  kb = []
  human = {}
  mortal = {}

  for name in constants:
    h, m = symbols(f"Human_{name} Mortal_{name}")
    human[name] = h
    mortal[name] = m
    kb.append(Implies(h,m))
  return kb, human, mortal

In [28]:
constants = ["Socrates", "Plato"]

kb_people, human, mortal = make_human_mortal_kb(constants)

# Add facts
kb_people.append(human["Socrates"])
kb_people.append(human["Plato"])

#Query: Is plato mortal?

check_entailment(
    kb_people,
    mortal["Plato"],
    "Grounded FOL with multiple constants: Is plato mortal? "
)

Grounded FOL with multiple constants: Is plato mortal? 
Results: Entailment holds.
----------------------------------------------------------------------------------------------------


True

## Guide Questions and Answers

### 1. What is the difference between syntax and semantics?

**Answer:** Syntax refers to the structural rules and grammar used to write symbols and statements correctly, while semantics defines the actual meaning behind those symbols and statements.

### 2. Why is `P -> Q` true when `P` is false?

**Answer:** In logic, a conditional statement is considered automatically true if the starting condition is false, since a false premise cannot lead to a broken promise.

### 3. What does it mean for a knowledge base to entail a conclusion?

**Answer:** Entailment means that whenever all the statements in the knowledge base are true, the conclusion must also be true.

### 4. How does theorem proving use satisfiability checking?

**Answer:** Theorem proving can check if a conclusion is correct by turning the problem into a satisfiability test to see if there is any scenario where the premises are true and the conclusion is false.

### 5. What is one limitation of propositional logic compared to First-Order Logic?

**Answer:** Propositional logic cannot directly talk about objects, properties, or relations using variables and quantifiers, making it much harder to express general rules like "all humans are mortal."

## Reflection

### Challenges Encountered

* Translating complex English logic statements into precise propositional symbols without changing their underlying meaning.
* Understanding why conditional statements evaluate to true when the starting premise is false.

### What I Learned

* How to ground First-Order Logic by converting atomic statements into manageable propositional symbols.
* The distinct difference between syntax (structural rules) and semantics (the actual meaning behind the logic).